In [1]:
%load_ext autoreload
%autoreload 2

**Author:** Salvador Navas  
**Date:** 2025-06-27

In [2]:
# ipyleaflet / ipywidgets NOT required for AemetCSVLoader
# AEMETDownloader widget requires ipyleaflet — skipped in this demo
from pyhydra.data_sources.rainfall import AemetCSVLoader
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np
import tempfile, os
print('Imports OK')

Imports OK


# AEMET Daily Climate Data

## General Description

This tool automates the download of **daily meteorological observations** from the **AEMET OpenData API**.

---

## Why Pre-Downloading is Necessary

- Data is requested in time blocks for all stations.
- The API returns temporary download links which expire shortly.
- Downloaded data is saved as NetCDF and then extracted to CSV.

---

## AemetCSVLoader

Once the series CSVs are available (from `AEMETDownloader`), use `AemetCSVLoader` to:
- Load station metadata (`selected_stations.csv`)
- Load time series for any variable (`AEMET_<id>_series.csv`)
- Analyze data quality per station

> This notebook uses **synthetic demo data** that reproduces the CSV format without requiring API access.

In [3]:
# === SYNTHETIC DEMO DATA ===
# Creates AEMET_*_series.csv files in the format expected by AemetCSVLoader

TMPDIR = tempfile.mkdtemp()
rng = np.random.default_rng(42)

stations = ['1124E', '1152C', '1159', '2374X']
dates = pd.date_range('2012-01-01', '2024-12-31', freq='D')

# Write one CSV per station
for sid in stations:
    prec = rng.exponential(2.5, len(dates))
    prec[prec < 0.2] = 0.0   # dry days
    # Introduce ~3% missing
    missing_idx = rng.choice(len(dates), size=int(0.03 * len(dates)), replace=False)
    prec[missing_idx] = np.nan
    tmed = 15 + 10 * np.sin(2 * np.pi * (dates.dayofyear - 15) / 365) + rng.normal(0, 1.5, len(dates))
    df_s = pd.DataFrame({
        'time': dates,
        f'{sid}_prec': prec,
        f'{sid}_tmed': tmed,
    })
    df_s.to_csv(os.path.join(TMPDIR, f'AEMET_{sid}_series.csv'), index=False)

# Write selected_stations.csv (station metadata)
meta_df = pd.DataFrame({
    'idema': stations,
    'nombre': ['Madrid Retiro', 'Toledo', 'Guadalajara', 'Cuenca'],
    'lat': [40.41, 39.88, 40.63, 40.07],
    'lon': [-3.68, -4.05, -3.17, -2.13],
    'alt': [667, 515, 685, 995],
})
meta_df.to_csv(os.path.join(TMPDIR, 'selected_stations.csv'), index=False)

print(f'Synthetic AEMET data created in: {TMPDIR}')
print(f'Files: {os.listdir(TMPDIR)}')

Synthetic AEMET data created in: /var/folders/44/dw7p6q9108xcc4mmh_f7q1vc0000gn/T/tmpjwiodur6
Files: ['AEMET_1124E_series.csv', 'selected_stations.csv', 'AEMET_1159_series.csv', 'AEMET_2374X_series.csv', 'AEMET_1152C_series.csv']


---
## 1. Load station metadata

In [4]:
loader = AemetCSVLoader(TMPDIR)
stations_df = loader.load_station_data()
print('Station metadata:')
stations_df

Station metadata:


,idema,nombre,lat,lon,alt
0,1124E,Madrid Retiro,40.41,-3.68,667
1,1152C,Toledo,39.88,-4.05,515
2,1159,Guadalajara,40.63,-3.17,685
3,2374X,Cuenca,40.07,-2.13,995


---
## 2. Load time series for precipitation

In [5]:
series_df = loader.load_series_data('prec')
print(f'Shape: {series_df.shape}')
print(f'Columns (stations): {list(series_df.columns)}')
series_df.head()

Shape: (4749, 4)
Columns (stations): ['1124E', '1159', '2374X', '1152C']


,1124E,1159,2374X,1152C
time,,,,
2012-01-01,6.010522,0.254554,10.171772,4.863614
2012-01-02,5.840474,5.727551,0.383648,0.663696
2012-01-03,5.961902,3.525033,2.503400,3.839159
2012-01-04,0.699486,3.600919,2.394347,3.741871
2012-01-05,0.216093,1.460824,0.862380,0.888203


---
## 3. Data quality analysis

In [6]:
quality_df = loader.analyze_series_quality()
print('Quality metrics per station:')
quality_df

Quality metrics per station:


,station_id,start_year,end_year,missing_percent,full_years,full_months,min_value,max_value
0,1124E,2012,2024,2.99,0,143,0.0,23.795299
1,1159,2012,2024,2.99,0,146,0.0,26.231366
2,2374X,2012,2024,2.99,0,144,0.0,19.416258
3,1152C,2012,2024,2.99,0,148,0.0,18.154001


---
## 4. Plot

In [7]:
fig, axes = plt.subplots(2, 1, figsize=(14, 7))

# Monthly mean precipitation per station
monthly = series_df.resample('ME').sum()
for col in monthly.columns:
    axes[0].plot(monthly.index, monthly[col], lw=0.7, alpha=0.8, label=col)
axes[0].set_ylabel('Monthly precipitation (mm)')
axes[0].set_title('AEMET Stations — Monthly Precipitation (synthetic demo)', fontsize=13)
axes[0].legend(fontsize=8)
axes[0].grid(alpha=0.3)

# Seasonal regime — average of all stations
seasonal = series_df.groupby(series_df.index.month).mean().mean(axis=1)
axes[1].bar(seasonal.index, seasonal.values, color='steelblue', alpha=0.85)
axes[1].set_xticks(range(1, 13))
axes[1].set_xticklabels(['Jan','Feb','Mar','Apr','May','Jun',
                          'Jul','Aug','Sep','Oct','Nov','Dec'])
axes[1].set_ylabel('Mean daily precipitation (mm)')
axes[1].set_title('Seasonal regime — all stations average')
axes[1].grid(alpha=0.3, axis='y')

plt.tight_layout()
plt.savefig('/tmp/AEMET_download.png', dpi=100, bbox_inches='tight')
plt.close()
print('Plot saved to /tmp/AEMET_download.png')

Plot saved to /tmp/AEMET_download.png